## Revision 3 — Python 3.13 / JAX compatibility

# GA–DKL–PyBaMM: voltage waveform discovery
**Research notebook | voltage-controlled charging | time vs lithium plating**

We compare constant voltage, CC–CV, random search, direct waveform GA, and DKL-assisted GA at equal simulator-call budgets. The genotype is the sampled voltage waveform, not coefficients. Voltage is prescribed and current is solved by the DFN. Piecewise constant holds are represented by short linear transitions (1 s) to avoid mathematical discontinuities in a single drive-cycle step. The final target is a calibrated negative-electrode volume-average concentration corresponding to 80% initial SOC, a proxy for stored charge rather than total injected charge. This is a model-based benchmark, not a validated charging prescription.

Run top-to-bottom in Colab. Start with one run and a small budget; increase to 20 runs for publication. No precomputed results are supplied.

In [ ]:
import sys, subprocess, os
print('Revision 3 | Python:', sys.version)
if any(k in sys.modules for k in ('jax','jaxlib','pybamm')):
    raise RuntimeError('Use Runtime > Disconnect and delete runtime, then reconnect and run all.')
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'jax', 'jaxlib'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pybamm==25.12.2'])
import importlib.util
if importlib.util.find_spec('jax') is not None:
    raise RuntimeError('JAX remains installed; use a clean dedicated runtime.')
os.environ['PYBAMM_DISABLE_TELEMETRY'] = 'true'
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
import pybamm, time, json, pathlib
from scipy.stats import norm
from scipy.integrate import trapezoid
from IPython.display import display
assert pybamm.__version__ == '25.12.2', pybamm.__version__
pybamm.set_logging_level('ERROR')
torch.set_num_threads(1)
print('Revision 3 ready:', pybamm.__version__, torch.__version__)


## 1. Configuration
The Lagrangian scalarization is $J=t/t_0+\lambda P/P_0$. The plating proxy $P$ is the maximum increase in the model's plated/dead-lithium equivalent capacity. Final capacity loss and current are also reported. Infeasible solutions are always ranked below feasible solutions; no arbitrary smoothness penalty is imposed. Voltage and current bounds are model-study assumptions, not certified cell limits.

In [ ]:
N_RUNS = 1                 # change to 20 (or any positive integer)
BUDGET = 30                # simulator calls per method per run, includes initial design
N_INIT = 10
POOL = 128
N_SEG = 24
HORIZON = 10800.0
RAMP = 1.0
V_MIN, V_MAX = 3.4, 4.2
I_MAX = 15.0
SOC0, SOC_TARGET = .2, .8
TEMP = 298.15
LAMBDA = 1.0               # repeat with 0, .1, 1, 10, etc.
T_REF, P_REF = 3600., .01  # s, Ah (scales, not tolerances)
MODEL = 'DFN'              # optionally SPMe; verify model compatibility
BASE_SEED = 42
OUT = pathlib.Path('ga_dkl_pybamm_results'); OUT.mkdir(exist_ok=True)
assert N_RUNS >= 1 and BUDGET >= N_INIT >= 6 and N_SEG >= 3
assert 0 < RAMP < HORIZON/N_SEG
CONC = 'Negative electrode volume-averaged concentration [mol.m-3]'
PLATE = 'Loss of capacity to negative lithium plating [A.h]'

def make_model(kind=MODEL):
    return getattr(pybamm.lithium_ion, kind)(options={'lithium plating':'partially reversible'})
def params():
    p = pybamm.ParameterValues('OKane2022')
    p.update({'Ambient temperature [K]':TEMP,'Initial temperature [K]':TEMP,
              'Upper voltage cut-off [V]':4.21})
    return p

def solver(): return pybamm.IDAKLUSolver(rtol=1e-6, atol=1e-8)
def series(sol, name):
    a = np.asarray(sol[name].entries).squeeze()
    if a.ndim != 1 or len(a) != len(sol.t): raise ValueError(f'Unexpected shape {name}: {a.shape}')
    return a

def calibrate(soc):
    m = make_model(); p=params()
    exp=pybamm.Experiment([pybamm.step.current(0.,duration=1.,period=1.)])
    sol=pybamm.Simulation(m,parameter_values=p,experiment=exp,solver=solver()).solve(initial_soc=soc)
    return float(series(sol,CONC)[0]),float(series(sol,'Voltage [V]')[0])
C0, OCV0 = calibrate(SOC0)
CT, OCVT = calibrate(SOC_TARGET)
assert CT > C0
print('Calibration:', C0, CT, OCV0, OCVT)

## 2. Simulator and objective
A custom event terminates the voltage drive cycle at the target stored-charge proxy. The CC–CV reference uses the same target event on both steps; analysis truncates at the first target crossing. Current magnitude is checked on all recorded pre-target samples (not a continuous-time guarantee). Solver failures are recorded explicitly. The baseline and all optimizers use the same initial state and physical model.

In [ ]:
def drive(v):
    v=np.clip(np.asarray(v,float),V_MIN,V_MAX)
    edges=np.linspace(0,HORIZON,N_SEG+1)
    ts=[0.]; vs=[v[0]]
    for k in range(1,N_SEG):
        ts.extend([edges[k]-RAMP,edges[k]])
        vs.extend([v[k-1],v[k]])
    ts.append(HORIZON); vs.append(v[-1])
    return np.column_stack((ts,vs))

def termination():
    return pybamm.step.CustomTermination('Stored charge target',lambda variables: CT-variables[CONC])

def evaluate(v=None, cc=None, trace=False):
    start=time.perf_counter()
    try:
        if cc is None:
            steps=[pybamm.step.voltage(drive(v),duration=HORIZON,period=2.,termination=termination(),skip_ok=False)]
        else:
            steps=[pybamm.step.current(-float(cc),duration=HORIZON,period=2.,termination=['4.2 V',termination()],skip_ok=False),
                   pybamm.step.voltage(4.2,duration=HORIZON,period=2.,termination=termination(),skip_ok=True)]
        sol=pybamm.Simulation(make_model(),parameter_values=params(),solver=solver(),experiment=pybamm.Experiment([tuple(steps)])).solve(initial_soc=SOC0)
        t=np.asarray(sol.t); C=series(sol,CONC); I=series(sol,'Current [A]'); V=series(sol,'Voltage [V]'); P=series(sol,PLATE)
        if not np.all(np.isfinite(np.stack([t,C,I,V,P]))): raise ValueError('nonfinite solution')
        # Target event is root-solved; tolerance accommodates event numerical accuracy.
        hit=np.flatnonzero(C >= CT-1e-5*(CT-C0))
        k=int(hit[0]) if len(hit) else len(t)-1
        # CC–CV can run beyond the global horizon; evaluate only within it.
        k=min(k,max(0,np.searchsorted(t,HORIZON,side='right')-1))
        t,C,I,V,P=[a[:k+1] for a in (t,C,I,V,P)]
        progress=float((C[-1]-C0)/(CT-C0)); peak=float(max(0,np.max(P-P[0])))
        elapsed=float(t[-1]); imax=float(np.max(np.abs(I)))
        violation=max(0,1-progress)+max(0,imax/I_MAX-1)+max(0,(np.max(V)-V_MAX)/.1)+max(0,(V_MIN-np.min(V))/.1)
        feasible=bool(progress>=1-1e-5 and imax<=I_MAX+1e-3 and np.max(V)<=V_MAX+1e-5 and np.min(V)>=V_MIN-1e-5)
        J=elapsed/T_REF+LAMBDA*peak/P_REF
        # Monotone bounded mapping separates feasible and infeasible observations for DKL.
        score=J/(1+J) if feasible else 1+violation
        row=dict(feasible=feasible,score=score,J=J,time_s=elapsed,peak_Ah=peak,final_Ah=float(P[-1]-P[0]),progress=progress,max_A=imax,net_input_Ah=float(-trapezoid(I,t)/3600),error='',wall_s=time.perf_counter()-start)
        if trace: row['trace']=dict(t=t,C=C,I=I,V=V,P=P)
        return row
    except Exception as e:
        return dict(feasible=False,score=10.,J=np.nan,time_s=np.nan,peak_Ah=np.nan,final_Ah=np.nan,progress=np.nan,max_A=np.nan,net_input_Ah=np.nan,error=f'{type(e).__name__}: {e}',wall_s=time.perf_counter()-start)

# Smoke test: verify model variables, solver and event before optimization.
smoke=evaluate(np.linspace(max(V_MIN,OCV0+.02),4.2,N_SEG),trace=True)
print({k:v for k,v in smoke.items() if k!='trace'})
if smoke['error']: raise RuntimeError(smoke['error'])

## 3. Baselines and physical response
Constant voltage and conventional CC–CV are independent reference protocols. They are not included in the optimizer-call budget except where an identical waveform is explicitly part of the common initial design. Inspect the current spikes and model validity before launching a large campaign.

In [ ]:
base=[]; traces={}
for v in [3.8,3.9,4.0,4.1,4.2]:
    name=f'CV {v:.2f} V'; r=evaluate(np.full(N_SEG,v),trace=True); traces[name]=r.pop('trace',None); base.append(dict(name=name,**r))
for a in [2.5,5.,10.]:
    name=f'CC-CV {a:g} A'; r=evaluate(cc=a,trace=True); traces[name]=r.pop('trace',None); base.append(dict(name=name,**r))
baselines=pd.DataFrame(base); display(baselines); baselines.to_csv(OUT/'baselines.csv',index=False)
def plot_traces(traces):
    fig,ax=plt.subplots(2,2,figsize=(12,7))
    for name,d in traces.items():
        if d is None: continue
        x=d['t']/60
        for a,y in zip(ax.flat,[d['V'],-d['I'],SOC0+(SOC_TARGET-SOC0)*(d['C']-C0)/(CT-C0),1000*(d['P']-d['P'][0])]): a.plot(x,y,label=name)
    for a,y in zip(ax.flat,['Voltage (V)','Charging current (A)','Stored-charge SOC proxy','Plating capacity (mAh)']): a.set(xlabel='Time (min)',ylabel=y); a.grid(alpha=.2)
    ax[0,0].legend(fontsize=7); fig.tight_layout(); return fig
plot_traces(traces); plt.show()

## 4. Direct waveform genetic operators
Gaussian spike, step mutation, segment replacement, one-point crossover, two-point crossover, and smooth crossover. No penalty for jump magnitude or number. Random search draws independent waveforms from the same initialization family. Bounds are applied to every offspring. The fixed horizon is a maximum; actual charging time is determined by the event.

In [ ]:
def random_wave(rng):
    x=np.arange(N_SEG); a=rng.uniform(max(V_MIN,OCV0-.03),min(V_MAX,OCV0+.18)); b=rng.uniform(4.0,V_MAX)
    if rng.random()<.2: return np.full(N_SEG,rng.uniform(3.8,V_MAX))
    y=np.interp(x,np.linspace(0,N_SEG-1,5),np.linspace(a,b,5)+rng.normal(0,.07,5))
    return np.clip(y,V_MIN,V_MAX)
def initial(rng):
    return [np.full(N_SEG,v) for v in [3.9,4.0,4.1,4.2]]+[np.linspace(np.clip(OCV0+.025,V_MIN,V_MAX),4.2,N_SEG),np.linspace(np.clip(OCV0+.08,V_MIN,V_MAX),4.2,N_SEG)]+[random_wave(rng) for _ in range(N_INIT-6)]
def mutate(a,rng,op=None):
    x=np.arange(N_SEG); y=a.copy(); i,j=sorted(rng.choice(N_SEG+1,2,replace=False)); op=op or rng.choice(['gaussian','step','replace'])
    if op=='gaussian': y+=rng.normal(0,.09)*np.exp(-.5*((x-rng.uniform(0,N_SEG-1))/rng.uniform(.5,5))**2)
    elif op=='step': y[i:]+=rng.normal(0,.08)
    elif op=='replace': y[i:j]=random_wave(rng)[i:j]
    else: raise ValueError(op)
    return np.clip(y,V_MIN,V_MAX)
def cross(a,b,rng,op=None):
    op=op or rng.choice(['one','two','smooth']); i,j=sorted(rng.choice(np.arange(1,N_SEG),2,replace=False)); y=a.copy()
    if op=='one': y[i:]=b[i:]
    elif op=='two': y[i:j]=b[i:j]
    elif op=='smooth':
        w=1/(1+np.exp(-(np.arange(N_SEG)-i)/rng.uniform(.5,3))); y=(1-w)*a+w*b
    else: raise ValueError(op)
    return np.clip(y,V_MIN,V_MAX)
rng=np.random.default_rng(BASE_SEED); a,b=initial(rng)[4:6]
fig,ax=plt.subplots(2,3,figsize=(12,6))
for p,op in zip(ax.flat,['gaussian','step','replace','one','two','smooth']):
    y=mutate(a,rng,op) if op in ['gaussian','step','replace'] else cross(a,b,rng,op)
    p.step(np.arange(N_SEG),a,where='post',alpha=.4); p.step(np.arange(N_SEG),y,where='post'); p.set(title=op,ylabel='V',xlabel='Segment')
fig.tight_layout(); plt.show()

## 5. Deep kernel learning
An MLP learns a four-dimensional embedding jointly with the hyperparameters of an exact RBF Gaussian process by minimizing negative log marginal likelihood. Expected improvement selects a candidate from a pool of genetically generated waveforms. A fixed exploration floor and random injection reduce premature convergence. GP uncertainty is conditional on the fitted embedding and is not a calibrated Bayesian neural-network posterior.

In [ ]:
class DKL(nn.Module):
    def __init__(self):
        super().__init__(); self.encoder=nn.Sequential(nn.Linear(N_SEG,32),nn.Tanh(),nn.Linear(32,4),nn.Tanh()).double()
        self.log_length=nn.Parameter(torch.tensor(-.2,dtype=torch.double)); self.log_amp=nn.Parameter(torch.tensor(0.,dtype=torch.double)); self.log_noise=nn.Parameter(torch.tensor(-4.,dtype=torch.double))
    def kernel(self,a,b):
        za,zb=self.encoder(a),self.encoder(b)
        d=torch.cdist(za,zb).square(); return self.log_amp.exp()*torch.exp(-.5*d/self.log_length.exp().square())
    def factor(self,x):
        K=self.kernel(x,x)+(self.log_noise.exp()+1e-6)*torch.eye(len(x),dtype=x.dtype); return torch.linalg.cholesky(K)
    def nll(self,x,y):
        L=self.factor(x); alpha=torch.cholesky_solve(y[:,None],L)
        return .5*(y[:,None]*alpha).sum()+L.diag().log().sum()+.001*sum(p.square().sum() for p in self.encoder.parameters())

def fit_predict(X,y,pool,seed):
    torch.manual_seed(int(seed)); model=DKL(); opt=torch.optim.Adam(model.parameters(),lr=.015)
    tensor=lambda x: torch.as_tensor(x,dtype=torch.double)
    x=tensor((np.asarray(X)-V_MIN)/(V_MAX-V_MIN)); z=tensor((np.asarray(pool)-V_MIN)/(V_MAX-V_MIN))
    center=float(np.mean(y)); scale=max(float(np.std(y)),1e-5); yy=tensor((np.asarray(y)-center)/scale)
    for _ in range(75):
        opt.zero_grad(); loss=model.nll(x,yy); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),10.); opt.step()
        with torch.no_grad():
            model.log_noise.clamp_(-8,1); model.log_length.clamp_(-3,3); model.log_amp.clamp_(-4,4)
    with torch.no_grad():
        L=model.factor(x); k=model.kernel(x,z); alpha=torch.cholesky_solve(yy[:,None],L); mu=(k.T@alpha).squeeze(1)
        v=torch.linalg.solve_triangular(L,k,upper=False); var=(model.log_amp.exp()-v.square().sum(0)).clamp_min(1e-9)
    return mu.numpy()*scale+center,var.sqrt().numpy()*scale

def ei(mu,sd,best):
    sd=np.maximum(sd,1e-9); delta=best-mu-.005; z=delta/sd
    return delta*norm.cdf(z)+sd*norm.pdf(z)
# Test surrogate independently of PyBaMM outcomes.
xx=np.array(initial(np.random.default_rng(1))); yy=xx.mean(1); mu,sd=fit_predict(xx,yy,xx,1)
assert np.isfinite(mu).all() and (sd>0).all()

## 6. Equal-budget optimization
All methods share the same initial design within each replicate. GA uses tournament selection from the evaluated archive, with one new physical evaluation per iteration. DKL-GA uses the same parent selection and genetic operators, but ranks a larger candidate pool with expected improvement. Random injection is shared. Each failed solver call consumes budget; no synthetic evaluations replace missing physics.

In [ ]:
def offspring(X,y,rng):
    def parent():
        ix=rng.choice(len(X),min(3,len(X)),replace=False); return X[ix[np.argmin(np.asarray(y)[ix])]]
    if rng.random()<.15: return random_wave(rng)
    a=parent(); b=parent(); child=cross(a,b,rng) if rng.random()<.7 else a.copy()
    return mutate(child,rng) if rng.random()<.8 else child

def campaign(method,seed,shared):
    rng=np.random.default_rng(seed+1000); X=[]; rows=[]
    def add(v):
        r=evaluate(v); X.append(np.asarray(v).copy()); rows.append(dict(method=method,seed=seed,call=len(X),waveform=json.dumps(v.tolist()),**r))
    for v,r in shared:
        X.append(v.copy()); rows.append(dict(method=method,seed=seed,call=len(X),waveform=json.dumps(v.tolist()),**r))
    while len(X)<BUDGET:
        y=[r['score'] for r in rows]
        if method=='Random': v=random_wave(rng)
        elif method=='GA': v=offspring(X,y,rng)
        else:
            pool=np.array([offspring(X,y,rng) for _ in range(POOL)])
            # Exclude exact duplicates when possible.
            mask=np.min(np.max(np.abs(pool[:,None,:]-np.asarray(X)[None,:,:]),axis=2),axis=1)>1e-8
            if mask.any(): pool=pool[mask]
            try:
                mu,sd=fit_predict(X,y,pool,seed+len(X)); v=pool[np.argmax(ei(mu,sd,min(y)))]
            except (RuntimeError,ValueError) as e:
                print('DKL fallback:',e); v=pool[0]
        add(v)
        if len(X)%5==0: print(seed,method,len(X),'best',round(min(r['score'] for r in rows),4),flush=True)
    return rows

allrows=[]
for run in range(N_RUNS):
    seed=BASE_SEED+run; init=initial(np.random.default_rng(seed)); shared=[(v,evaluate(v)) for v in init]
    for method in ['Random','GA','DKL-GA']:
        allrows.extend(campaign(method,seed,shared)); pd.DataFrame(allrows).to_csv(OUT/'evaluations.csv',index=False)
df=pd.DataFrame(allrows); display(df.groupby('method').agg(feasible_fraction=('feasible','mean'),solver_failures=('error',lambda x:(x!='').sum())))

## 7. Convergence, Pareto trade-off and final waveforms
The convergence curve shows the best feasible Lagrangian value after each evaluation; missing feasible incumbents remain NaN, not zero. For N=1 individual curves are shown; for N>1 the median and interquartile range are shown, conditional on feasibility, with feasibility fraction plotted separately. Pareto points are nondominated among evaluated feasible candidates only, not a proof of global optimality.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
for method,g in df.groupby('method'):
    curves=[]; flags=[]
    for seed,h in g.groupby('seed'):
        h=h.sort_values('call'); val=np.where(h.feasible,h.J,np.inf); best=np.minimum.accumulate(val); flags.append(np.isfinite(best)); curves.append(np.where(np.isfinite(best),best,np.nan))
    a=np.array(curves); x=np.arange(1,BUDGET+1); valid=np.any(np.isfinite(a),axis=0)
    if valid.any():
        med=np.nanmedian(a[:,valid],axis=0); line=ax[0].plot(x[valid],med,label=method)[0]
        if N_RUNS>1: ax[0].fill_between(x[valid],np.nanpercentile(a[:,valid],25,axis=0),np.nanpercentile(a[:,valid],75,axis=0),alpha=.2,color=line.get_color())
    ax[1].plot(x,np.mean(flags,axis=0),label=method)
ax[0].set(xlabel='Simulator calls',ylabel='Best feasible J'); ax[1].set(xlabel='Simulator calls',ylabel='Fraction with feasible incumbent',ylim=(-.03,1.03))
for p in ax: p.legend(); p.grid(alpha=.2)
fig.tight_layout(); fig.savefig(OUT/'convergence.png',dpi=180); plt.show()

def pareto(a):
    a=np.asarray(a); return np.array([not np.any(np.all(a<=v,axis=1)&np.any(a<v,axis=1)) for v in a],dtype=bool)
f=df[df.feasible].copy(); fig,ax=plt.subplots(figsize=(7,5))
for method,g in f.groupby('method'): ax.scatter(g.time_s/60,g.peak_Ah*1000,label=method,alpha=.5)
if len(f):
    front=f.loc[pareto(f[['time_s','peak_Ah']])].sort_values('time_s'); ax.plot(front.time_s/60,front.peak_Ah*1000,'k--',label='Observed Pareto front'); front.to_csv(OUT/'pareto.csv',index=False)
g=baselines[baselines.feasible]; ax.scatter(g.time_s/60,g.peak_Ah*1000,marker='x',label='Baselines'); ax.set(xlabel='Time to target (min)',ylabel='Peak plating (mAh)'); ax.legend(); fig.tight_layout(); fig.savefig(OUT/'pareto.png',dpi=180); plt.show()
if len(f):
    winners=f.loc[f.groupby(['method','seed']).J.idxmin()]; display(winners.drop(columns='waveform')); winners.to_csv(OUT/'winners.csv',index=False)
    best=f.loc[f.groupby('method').J.idxmin()]; tr={}
    for _,r in best.iterrows():
        q=evaluate(np.array(json.loads(r.waveform)),trace=True); tr[r.method]=q.get('trace')
    fig=plot_traces(tr); fig.savefig(OUT/'best_traces.png',dpi=180); plt.show()
else: print('No feasible candidates; inspect failures, current cap and horizon. Do not claim improvement.')

## 8. Interpretation and next experiments
- Repeat independent campaigns across penalty weights (e.g. 0, 0.1, 1, 10) to reveal different sections of the Pareto front. Save each weight to a separate output directory. A single scalarization does not recover a nonconvex Pareto front in general.
- Re-evaluate winners with a finer DFN mesh, tighter tolerances, finer recording and shorter transition ramps. Check instantaneous current peaks and plating near the separator. Evaluate actual irreversible dead lithium separately if needed: the capacity-loss variable is a combined inventory proxy, not gross deposition flux.
- Validate at multiple temperatures, initial SOCs and parameter sets. No experimental safety claim follows from this model alone.
- The common initial design is physically evaluated once per replicate and charged logically to all three budgets. Final trace replays are diagnostic calls outside the optimization budget. CC–CV baselines are likewise outside that budget.
- Compare against optimized low-dimensional monotonic or CC–CV protocols before claiming structural novelty. The present benchmark does not establish superiority of GA or DKL in advance.

References: [PyBaMM plating model](https://docs.pybamm.org/en/v25.8.0/source/examples/notebooks/models/lithium-plating.html), [experiment steps](https://docs.pybamm.org/en/v25.10.1/source/api/experiment/experiment_steps.html), [custom events](https://docs.pybamm.org/en/v25.8.0/source/examples/notebooks/simulations_and_experiments/custom-experiments.html), Wilson et al., Deep Kernel Learning (AISTATS 2016).